In [1]:
from utils.analytics import eval_fit_methods
import networks
import estimator.stable_estimators as se
from scipy.stats import levy_stable
import matplotlib.pyplot as plt 
import seaborn as sns 
from utils import plots, optimreg
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import pandas as pd
import pickle
import numpy as np

In [ ]:
import re

filename = r"/Users/janikwahrheit/Library/CloudStorage/OneDrive-Persönlich/01_Studium/01_Bachelor/Bachelorarbeit/Code/data/training_results/mnist_log.txt"
results = {}

with open(filename, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    line = line.strip()
    if line.startswith("Finished:"):

        match_model = re.search(r"Finished:\s*([^\|]+)", line)
        model = match_model.group(1).strip() if match_model else "unknown"
        

        match_reg = re.search(r"Reg:\s*([^\|]+)", line)
        reg = match_reg.group(1).strip() if match_reg else "None"
        

        match_opt = re.search(r"Optimizer:\s*([^\|]+)", line)
        optimizer = match_opt.group(1).strip() if match_opt else "None"
        
        key = f"{model}_{reg}_{optimizer}"

        acc_line = ""
        for j in range(i+1, min(i+5, len(lines))):  
            if lines[j].strip().startswith("Accuracies:"):
                acc_line = lines[j].strip()
                break
        

        acc_match = re.search(r"\[([0-9.,\s]+)\]", acc_line)
        if acc_match:
            acc_str = acc_match.group(1)
            acc_list = [float(x) for x in acc_str.split(",")]
            results[key] = acc_list


for k, v in results.items():
    print(k, v)

fc3_mnist_None_sgd [0.908, 0.9087, 0.9097, 0.9094, 0.9122, 0.9115, 0.9127, 0.9066, 0.9117, 0.9088]
fc3_mnist_lasso_sgd [0.9097, 0.9075, 0.9084, 0.9071, 0.9093, 0.9063, 0.9098, 0.9062, 0.9101, 0.9061]
fc3_mnist_hill_sgd [0.8795, 0.8825, 0.8888, 0.8934, 0.8888, 0.8886, 0.9145, 0.9101, 0.899, 0.9094]
fc3_mnist_parabolic_hill_sgd [0.7886, 0.4615, 0.8852, 0.7109, 0.8621, 0.9022, 0.6777, 0.1814, 0.3016, 0.165]
fc3_mnist_xiao_sgd [0.8121, 0.8203, 0.8106, 0.8154, 0.8273, 0.8212, 0.8141, 0.8183, 0.8119, 0.8051]
fc3_mnist_decay_sgd [0.8927, 0.8933, 0.8923, 0.8937, 0.8944, 0.8968, 0.8914, 0.8894, 0.8909, 0.8911]
fc3_mnist_None_adam [0.9727, 0.9762, 0.9754, 0.9751, 0.9754, 0.9743, 0.9759, 0.9705, 0.9723, 0.9768]
fc3_mnist_lasso_adam [0.9675, 0.9709, 0.9721, 0.9678, 0.9707, 0.9727, 0.9637, 0.9738, 0.9697, 0.972]
fc3_mnist_hill_adam [0.9621, 0.9637, 0.9612, 0.9674, 0.962, 0.9611, 0.9668, 0.9574, 0.9566, 0.966]
fc3_mnist_parabolic_hill_adam [0.9688, 0.9682, 0.9687, 0.9636, 0.9704, 0.9703, 0.9737, 0.9

In [3]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  
])


full_train_dataset = datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

train_size = int(0.8 * len(full_train_dataset))  
val_size   = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])


batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [12]:
# Iterator aus dem DataLoader erstellen
train_iter = iter(train_loader)

# Ersten Batch holen
X_first, y_first = next(train_iter)

print("X_first shape:", X_first.shape)
print("y_first shape:", y_first.shape)
print(y)
print(X)


X_first shape: torch.Size([32, 784])
y_first shape: torch.Size([32])
tensor([9, 9, 0, 6, 3, 0, 2, 9, 1, 1, 4, 0, 2, 0, 7, 1, 0, 4, 6, 7, 5, 1, 9, 9,
        4, 4, 0, 0, 2, 8, 3, 0])
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])


In [ ]:
model = networks.FCNet(layer_sizes=[784] + [256]*3 + [10], activation='relu', weight_init='gaussian')
model.to(device)

optimizer = optimreg.get_optimizer(model, optimizer_name='sgd', lr=1e-3)

In [7]:
epochs = 5

model.to(device)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)

        for name, param in model.named_parameters(): 
            if "weight" not in name: 
                continue
            
            if param.grad is not None:
                w_l = param.grad.norm()  
            else:
                w_l = 1.0
            
            print(w_l)


        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)
    


1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
